In [77]:
import pandas as pd
import statsmodels.formula.api as smf
import scipy.stats as st
import matplotlib.pyplot as plt
from statsmodels.iolib.summary2 import summary_col
import numpy as np
from statsmodels.stats.diagnostic import het_breuschpagan

url = 'https://raw.githubusercontent.com/git-iush/large_family/refs/heads/main/data2.parquet'

In [78]:
df = pd.read_parquet(url)
df.head()

,SERIAL,HHWT,CLUSTER,STRATA,PERNUM,PERWT,NCHILD,SEX,AGE,MARST,...,BLACK,AIAN,API,OTHER,MULTR,HISP,EXPANSION,WELFARE,PRSNINC,YRSEDUC
0,4,4.0,2024000000041,280201,1,4.0,0,2,22,6,...,0,0,1,0,0,0,0,0.0,1000.0,13
1,9,74.0,2024000000091,280301,1,74.0,0,2,18,6,...,1,0,0,0,0,0,0,0.0,5100.0,12
2,12,55.0,2024000000121,160001,1,55.0,0,2,19,6,...,0,0,0,0,0,0,0,0.0,0.0,12
3,15,17.0,2024000000151,120201,1,17.0,0,2,19,6,...,0,0,0,0,0,0,0,0.0,6000.0,12
4,19,71.0,2024000000191,10001,1,71.0,0,2,23,6,...,0,0,0,0,0,0,0,0.0,0.0,13


In [79]:
df.columns

Index(['SERIAL', 'HHWT', 'CLUSTER', 'STRATA', 'PERNUM', 'PERWT', 'NCHILD',
       'SEX', 'AGE', 'MARST', 'HCOVANY', 'HINSCAID', 'EDUCD', 'INCTOT',
       'FTOTINC', 'INCWELFR', 'CONNECTICUT', 'MAINE', 'MASSACHUSETTS',
       'NEWHAMPSHIRE', 'RHODEISLAND', 'VERMONT', 'DELAWARE', 'NEWJERSEY',
       'NEWYORK', 'PENNSYLVANIA', 'ILLINOIS', 'INDIANA', 'MICHIGAN', 'OHIO',
       'WISCONSIN', 'IOWA', 'KANSAS', 'MINNESOTA', 'MISSOURI', 'NEBRASKA',
       'NORTHDAKOTA', 'SOUTHDAKOTA', 'VIRGINIA', 'ALABAMA', 'ARKANSAS',
       'GEORGIA', 'LOUISIANA', 'MISSISSIPPI', 'NORTHCAROLINA', 'SOUTHCAROLINA',
       'TEXAS', 'KENTUCKY', 'MARYLAND', 'OKLAHOMA', 'TENNESSEE',
       'WESTVIRGINIA', 'ARIZONA', 'COLORADO', 'IDAHO', 'MONTANA', 'NEVADA',
       'NEWMEXICO', 'UTAH', 'WYOMING', 'CALIFORNIA', 'OREGON', 'WASHINGTON',
       'ALASKA', 'HAWAII', 'DC', 'STATENAME', 'BLACK', 'AIAN', 'API', 'OTHER',
       'MULTR', 'HISP', 'EXPANSION', 'WELFARE', 'PRSNINC', 'YRSEDUC'],
      dtype='str')

In [80]:
medicaid_mapping = {1:0, 2:1}
df['MEDICAID'] = df['HINSCAID'].map(medicaid_mapping)
df['MEDFARE'] = df['MEDICAID'] * df['WELFARE']

df['FAMINC'] = df['FTOTINC'] - df['INCTOT']
df['PRSNINC'] = df['PRSNINC'] / 10000
df['FAMINC'] = df['FAMINC'] / 10000
df['PRSNINC2'] = df['PRSNINC'] ** 2
df['FAMINC2'] = df['FAMINC'] ** 2

df['AGE2'] = df['AGE'] ** 2

df['HIGHSCHOOL'] = [1 if i == 63 or i==64 else 0 for i in df['EDUCD']]
df['BACHELORS'] = [1 if i == 101 else 0 for i in df['EDUCD']]
df['ADVANCED'] = [1 if i>=114 else 0 for i in df['EDUCD']]

df['LARGEFAM'] = [1 if i >= 3 else 0 for i in df['NCHILD']]

In [81]:
ols_exp = 'NCHILD ~ MEDICAID + WELFARE + MEDFARE + AGE + AGE2 + PRSNINC + PRSNINC2 + FAMINC + FAMINC2 + C(MARST) + BLACK + AIAN + API + OTHER + MULTR + HISP + EXPANSION + HIGHSCHOOL + BACHELORS + ADVANCED'
ols_fe = model_exp = 'NCHILD ~ MEDICAID + WELFARE + MEDFARE + AGE + AGE2 + PRSNINC + PRSNINC2 + FAMINC + FAMINC2 + C(MARST) + BLACK + AIAN + API + OTHER + MULTR + HISP + C(STATENAME, Treatment("FLORIDA")) + HIGHSCHOOL + BACHELORS + ADVANCED'

In [82]:
reg_exp = smf.ols(ols_exp, data=df).fit()
reg_fe = smf.ols(ols_fe, data=df).fit()

In [83]:
print(reg_exp.summary())

                            OLS Regression Results                            
Dep. Variable:                 NCHILD   R-squared:                       0.333
Model:                            OLS   Adj. R-squared:                  0.333
Method:                 Least Squares   F-statistic:                 1.414e+04
Date:                Mon, 16 Nov 2026   Prob (F-statistic):               0.00
Time:                        18:05:07   Log-Likelihood:            -9.5785e+05
No. Observations:              678476   AIC:                         1.916e+06
Df Residuals:                  678451   BIC:                         1.916e+06
Df Model:                          24                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept        -1.6723      0.016   -107.028

In [84]:
print(reg_fe.summary())

                            OLS Regression Results                            
Dep. Variable:                 NCHILD   R-squared:                       0.337
Model:                            OLS   Adj. R-squared:                  0.337
Method:                 Least Squares   F-statistic:                     4715.
Date:                Mon, 16 Nov 2026   Prob (F-statistic):               0.00
Time:                        18:05:07   Log-Likelihood:            -9.5623e+05
No. Observations:              678476   AIC:                         1.913e+06
Df Residuals:                  678402   BIC:                         1.913e+06
Df Model:                          73                                         
Covariance Type:            nonrobust                                         
                                                          coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------

states = df['STATENAME'].unique()

expreg = {}

for s in states:
    ols_exp = 'NCHILD ~ MEDICAID + WELFARE + MEDFARE + AGE + AGE2 + PRSNINC + PRSNINC2 + FAMINC + FAMINC2 + MARST + BLACK + AIAN + API + OTHER + MULTR + HISP + EXPANSION + HIGHSCHOOL + BACHELORS + ADVANCED'

    ols = smf.ols(ols_exp, data=df[df['STATENAME']==s]).fit()
    expreg[s] = ols

results = summary_col(
    results = [reg_exp] + [expreg[i] for i in states],
    model_names = ['All'] + [i.capitalize() for i in states],
    info_dict = {'No. obs': lambda x: f'{int(x.nobs)}'},
    stars=True,
    regressor_order = ['Intercept', 'MEDFARE'],)

results.tables[0].drop('R-squared Adj.', inplace=True)
results.tables[0] = results.tables[0].transpose()
print(results)

In [85]:
results = summary_col(
    [reg_exp, reg_fe],
    model_names=['Expansion', 'State FE'],
    stars=True,
    float_format='%0.3f',
    info_dict={
        'Observations': lambda x: f'{int(x.nobs):,}',
        'R-squared': lambda x: f'{x.rsquared:.3f}'
    },
    drop_omitted=True
)

print(results)


                                                    Expansion  State FE
-----------------------------------------------------------------------
Intercept                                           -1.672*** -1.828***
                                                    (0.016)   (0.016)  
C(MARST)[T.2]                                       -0.911*** -0.898***
                                                    (0.009)   (0.009)  
C(MARST)[T.3]                                       -0.277*** -0.269***
                                                    (0.010)   (0.010)  
C(MARST)[T.4]                                       -0.495*** -0.490***
                                                    (0.005)   (0.005)  
C(MARST)[T.5]                                       -0.401*** -0.396***
                                                    (0.016)   (0.016)  
C(MARST)[T.6]                                       -0.924*** -0.910***
                                                    (0.003)   (

print(df[['PRSNINC', 'FAMINC']].describe(
    percentiles=[.01, .25, .50, .75, .99]
))

print(df[['PRSNINC', 'FAMINC']].head(10))